In [50]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pprint import pprint
from tabulate import tabulate

In [30]:
def append_to_dict(dictionary, key, item):
    """
    To append an item to a list in a dictionary.
    Parameters:
        dictionary (dict): The dictionary to append to.
        key (str): The key for the list to append to.
        item (Any): The item to append.
    """
    if key in dictionary.keys():
        dictionary[key].append(item)
    else:
        dictionary.update({key : [item]})

In [35]:
Run_Number, Min_Time, Max_Time, Zero_Time = np.loadtxt("Normalization/MegaHighStat.timewindows.csv", delimiter = ",", unpack = True, skiprows = 1)
Run_Number = [str(int(item)) for item in Run_Number]

for i in range (0, len(Run_Number)):
    print(int(Run_Number[i]), " -- ", Min_Time[i], " -- " ,Max_Time[i])

70251  --  32004.655076  --  32516.670566
70251  --  33025.724451  --  33537.73996
70251  --  33808.55257  --  33936.565608
70251  --  34366.409992  --  34878.425498
70251  --  34944.079165  --  35072.092201
70267  --  25384.222904  --  25896.238085
70267  --  26099.210933  --  26611.226129
70267  --  26742.593462  --  26870.606424
70267  --  27380.888406  --  27892.90361
70267  --  27972.884533  --  28100.897495
70386  --  18093.062105  --  18605.077318
70386  --  18859.674175  --  20184.30402
70428  --  13091.206605  --  13603.221888
70428  --  13689.855859  --  14202.471147
70428  --  14259.870842  --  14772.48613
70428  --  14869.923613  --  15381.9389
70428  --  15429.192727  --  15429.79473
70428  --  15582.252609  --  16094.867889
70428  --  16180.780675  --  16693.395951
70428  --  17485.606037  --  17998.221296
70428  --  18230.319319  --  18742.934585
70428  --  18950.604627  --  18951.20663
70428  --  19190.769588  --  19191.371591
70428  --  19223.89562  --  19736.510865
70

In [36]:
dataset = pd.read_csv("Normalization/MegaHighStat.vertex.csv")
dataset.head()

,Run Number,Event Number,Plot Time (Time axis of TAPlot),Detector Time (detectors internal clock),OfficialTime (run time),CutsType0,CutsType1,CutsType2,CutsType3,Vertex Status,X,Y,Z,Number of Helices,Number of Tracks
0,70251,531851,0.224206,32004.776214,32004.879282,1,0,0,0,1,-4.721640,-3.385894,0.782602,-1,2
1,70251,531852,0.542490,32005.094498,32005.197566,1,0,0,0,1,3.737765,-10.354581,-15.311709,-1,2
2,70251,531853,0.684592,32005.236599,32005.339668,1,0,0,0,1,-5.746450,5.236381,-13.752816,-1,2
3,70251,531854,0.843966,32005.395973,32005.499042,0,0,0,0,0,-99.000000,-99.000000,-99.000000,-1,1
4,70251,531855,1.051941,32005.603947,32005.707017,1,0,0,0,1,-2.140344,4.177726,14.984424,-1,2


In [72]:
runList = ["70251", 
           "70267", 
           "70386", 
           #"70428", 
           "70461", 
           "70471"]

" first element:  dump index"
" second element: start time"
" third element:  stop time"
lowpower = {
    "70251": [0],
    "70267": [0],
    "70386": [0],
    #"70428": [3],
    "70461": [0],
    "70471": [0],
}

Power = {
    "70251": ["12dB", "bypassing amplifier", "18dB x 8s  + 18dB x 4s", "18dB x 8s + 18dB x 4s"],
    "70267": ["12dB", "bypassing amplifier", "12dB x 8s + 18dB x 4s", "12dB x 8s + 18dB x 4s"],
    "70386": ["18dB", "--", "18dB x 8s + 18dB x 8s","15dB x 8s + 18dB x 8s"],
    #"70428": ["18dB", "bypassing amplifier", ""],
    "70461": ["12dB", "--", "12dB x 8s + 18dB x 8s", "12dB x 8s + 18dB x 8s"],
    "70471": ["12dB", "--", "12dB x 8s + 18dB x 8s", "12dB x 8s + 18dB x 8s"],
}

# get time for low power sweep
for index, run in enumerate(runList):
    dump_index = lowpower[run][0]
    mask = (np.array(Run_Number) == run)
    tmp = Min_Time[mask][dump_index]
    append_to_dict(lowpower, run, tmp)
    tmp = Max_Time[mask][dump_index]
    append_to_dict(lowpower, run, tmp)
    
    tmp = Max_Time[mask][-1]
    append_to_dict(lowpower, run, tmp)

pprint(lowpower)

{'70251': [0, 32004.655076, 32516.670566, 35072.092201],
 '70267': [0, 25384.222904, 25896.238085, 28100.897495],
 '70386': [0, 18093.062105, 18605.077318, 20184.30402],
 '70461': [0, 14967.566759, 15479.581954, 17045.852252],
 '70471': [0, 14083.433229, 14595.448435, 16185.051713]}


In [73]:
highpower_sweep = {}
lowpower_sweep = {}

for key, value in lowpower.items():
    classifier = "CutsType1"
    mask_run  = (dataset["Run Number"] == int(key))
    mask_time_low = (dataset["OfficialTime (run time)"] > value[1]) & (dataset["OfficialTime (run time)"] < value[2])
    mask_classifier = (dataset[classifier] == 1)
    mask_time_high = (dataset["OfficialTime (run time)"] > value[2])
    
    #print(mask_time.sum())
    #print(mask_run.sum())
    #print(mask_classifier.sum())
    total = (mask_time_low & mask_classifier & mask_run).sum() - 0.046 * (value[2] - value[1])
    append_to_dict(lowpower_sweep, key, total)

    total = (mask_time_high & mask_classifier & mask_run).sum() - 0.046 * (value[3] - value[2])
    append_to_dict(highpower_sweep, key, total)
    
pprint(lowpower_sweep)
pprint(highpower_sweep)

{'70251': [64.44728745999993],
 '70267': [36.44730167399989],
 '70386': [45.44730020200006],
 '70461': [24.447301030000002],
 '70471': [25.447300524]}
{'70251': [4182.45060479],
 '70267': [4742.58566714],
 '70386': [4503.355571708],
 '70461': [5308.951566292],
 '70471': [5300.878249212]}


In [75]:
for index, run in enumerate(runList):
    percentage = lowpower_sweep[run][0] / highpower_sweep[run][0]
    low = lowpower_sweep[run][0]; sigma_low = np.sqrt(low)
    high = highpower_sweep[run][0]; sigma_high = np.sqrt(high)
    
    dp = np.sqrt( sigma_low**2 / high**2 + sigma_high**2 * (low/high**2)**2)
    
    powerstring = "cb: " + Power[run][2] + " da: " + Power[run][3]
    print(f"{run} & {lowpower_sweep[run][0]:.1f} & {highpower_sweep[run][0]:.1f} & {100*percentage:.2} $ \pm $ {dp*100:.2}\\% & {powerstring} \\\\")

70251 & 64.4 & 4182.5 & 1.5 $ \pm $ 0.19\% & cb: 18dB x 8s  + 18dB x 4s da: 18dB x 8s + 18dB x 4s \\
70267 & 36.4 & 4742.6 & 0.77 $ \pm $ 0.13\% & cb: 12dB x 8s + 18dB x 4s da: 12dB x 8s + 18dB x 4s \\
70386 & 45.4 & 4503.4 & 1.0 $ \pm $ 0.15\% & cb: 18dB x 8s + 18dB x 8s da: 15dB x 8s + 18dB x 8s \\
70461 & 24.4 & 5309.0 & 0.46 $ \pm $ 0.093\% & cb: 12dB x 8s + 18dB x 8s da: 12dB x 8s + 18dB x 8s \\
70471 & 25.4 & 5300.9 & 0.48 $ \pm $ 0.095\% & cb: 12dB x 8s + 18dB x 8s da: 12dB x 8s + 18dB x 8s \\
